# GECS Task 1 — Cleaning v10 (Clean Architecture)

**Key change from v9c:**
- No ModelInput column — SegmentDescription is the target text column
- Empty rows filled with SegmentName + LongProfile
- All features stay in separate columns
- Text combination and sibling context built in Dataset class at training time
- Clean, API-friendly design

In [9]:
import os, warnings, re
from pathlib import Path
import numpy as np
import pandas as pd
try:
    import ftfy
except ImportError:
    raise ImportError('pip install ftfy')
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────
BASE_DIR   = Path.home() / 'Documents/capstone/depaul-morningstar-capstone'
RAW_DIR    = BASE_DIR / 'data' / 'raw'
OUTPUT_DIR = BASE_DIR / 'data' / 'cleaned_v10'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILE_T1     = RAW_DIR / 'task1_gecs_classification_final.csv'
FILE_HIER   = BASE_DIR / 'data' / 'cleaned_v6' / 'industries_Hierarchy.csv'
SPLITS_FILE = OUTPUT_DIR / 'canonical_splits.npz'

print('Path check:')
for f in [FILE_T1, FILE_HIER]:
    print(('  OK' if f.exists() else '  NOT FOUND'), f.name)
print(f'Output -> {OUTPUT_DIR}')

Path check:
  OK task1_gecs_classification_final.csv
  NOT FOUND industries_Hierarchy.csv
Output -> /Users/meetpatel/Documents/capstone/depaul-morningstar-capstone/data/cleaned_v10


In [10]:
# ── Load raw data ─────────────────────────────────────────
DTYPE_T1 = {
    'CompanyId'                  : str,
    'AsOfDate'                   : str,
    'LongProfile'                : str,
    'SegmentName'                : str,
    'SegmentDescription'         : str,
    'Revenue'                    : str,
    'total_revenue_company_as_of': str,
    'revenue_share'              : float,
    'is_largest_share_segment'   : str,
    'MstarGlobal'                : str,
}

raw_t1 = pd.read_csv(FILE_T1, dtype=DTYPE_T1, low_memory=False)
raw_t1['AsOfDate'] = pd.to_datetime(raw_t1['AsOfDate'], errors='coerce')

raw_t1['Revenue'] = pd.to_numeric(
    raw_t1['Revenue'].astype(str).str.replace(',', '').str.strip(),
    errors='coerce'
).fillna(0.0).clip(lower=0)

raw_t1['total_revenue_company_as_of'] = pd.to_numeric(
    raw_t1['total_revenue_company_as_of'].astype(str).str.replace(',', '').str.strip(),
    errors='coerce'
).fillna(0.0).clip(lower=0)

ORIGINAL_LEN = len(raw_t1)
print(f'Raw shape        : {raw_t1.shape}')
print(f'Unique companies : {raw_t1["CompanyId"].nunique():,}')
print(f'Target classes   : {raw_t1["MstarGlobal"].nunique()}')

Raw shape        : (53585, 10)
Unique companies : 23,207
Target classes   : 145


In [11]:
# ── Phase 2: Text Normalization ───────────────────────────
GENERIC_SEG_NAMES = {
    'other', 'others', 'all other', 'all other segments',
    'corporate', 'corporate and other', 'unallocated',
    'single segment', 'one segment', 'single',
    'n/a', 'na', 'none', '',
}

def normalize_text(text):
    if pd.isna(text) or str(text).strip() == '':
        return ''
    text = ftfy.fix_text(str(text))
    text = re.sub(r'[\u201c\u201d\u2018\u2019\u0022]', ' ', text)
    text = re.sub(r'\s*&\s*', ' and ', text)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    text = re.sub(r'[|~^`\\]', ' ', text)
    text = re.sub(r'\(\s*\)', ' ', text)
    text = re.sub(r'\(\s*\d+\s*\)', ' ', text)
    text = ' '.join(text.split()).lower()
    return text.strip()

def is_generic(s):
    return str(s).strip().lower() in GENERIC_SEG_NAMES if not pd.isna(s) else True

df_t1 = raw_t1.copy()
print('Normalizing text columns...')
for col in ['LongProfile', 'SegmentName', 'SegmentDescription']:
    df_t1[col] = df_t1[col].apply(normalize_text)
    empty = df_t1[col].str.strip().eq('').sum()
    print(f'  {col:25s} empty after normalize: {empty:,}')

# Flags
df_t1['segment_desc_imputed'] = df_t1['SegmentDescription'].str.strip().eq('')
df_t1['seg_name_generic']     = df_t1['SegmentName'].apply(is_generic)
df_t1['weak_signal_row']      = df_t1['segment_desc_imputed'] & df_t1['seg_name_generic']

# Short flags — numeric features for model
df_t1['lp_short_flag'] = (df_t1['LongProfile'].str.split().str.len() < 50).astype(int)
df_t1['sd_short_flag'] = (df_t1['SegmentDescription'].str.split().str.len() < 20).astype(int)
df_t1['sn_short_flag'] = (df_t1['SegmentName'].str.split().str.len() < 2).astype(int)

print(f'\nFlags:')
print(f'  segment_desc_imputed : {df_t1["segment_desc_imputed"].sum():,} ({df_t1["segment_desc_imputed"].mean()*100:.1f}%)')
print(f'  seg_name_generic     : {df_t1["seg_name_generic"].sum():,} ({df_t1["seg_name_generic"].mean()*100:.1f}%)')
print(f'  weak_signal_row      : {df_t1["weak_signal_row"].sum():,} ({df_t1["weak_signal_row"].mean()*100:.1f}%)')

Normalizing text columns...
  LongProfile               empty after normalize: 0
  SegmentName               empty after normalize: 0
  SegmentDescription        empty after normalize: 27,524

Flags:
  segment_desc_imputed : 27,524 (51.4%)
  seg_name_generic     : 7,992 (14.9%)
  weak_signal_row      : 6,268 (11.7%)


In [12]:
# ── Phase 3: Fix Empty SegmentDescription ─────────────────
# SIMPLE — no ModelInput column, no combined text blob
# Empty rows: SegmentName + LongProfile
# Non-empty rows: keep original SegmentDescription as-is

# ── Phase 3: Fix Empty SegmentDescription ─────────────────
def fix_segment_description(row):
    seg_desc = row['SegmentDescription'].strip()
    seg_name = row['SegmentName'].strip()
    long_p   = row['LongProfile'].strip()

    if seg_desc:
        return seg_desc

    if seg_name and not row['seg_name_generic']:
        return f"{seg_name} {long_p}"
    return long_p

len_before = len(df_t1)
df_t1['SegmentDescription'] = df_t1.apply(fix_segment_description, axis=1)
assert len(df_t1) == len_before

df_t1['seg_desc_len'] = df_t1['SegmentDescription'].str.len()
df_t1['token_est']    = (df_t1['seg_desc_len'] / 4.5).astype(int)

print('SegmentDescription fixed:')
print(f'  Row count unchanged: {len(df_t1):,} OK')
print(f'  Still empty        : {df_t1["SegmentDescription"].str.strip().eq("").sum():,}')
print()
print('Token distribution:')
for pct in [25, 50, 75, 90, 99]:
    print(f'  p{pct}: {df_t1["token_est"].quantile(pct/100):.0f} tokens')
print()
print('Samples:')
for i in [0, 1, 2]:
    row = df_t1.iloc[i]
    print(f'  [{row["MstarGlobal"]}] imputed={row["segment_desc_imputed"]}')
    print(f'  SD: {row["SegmentDescription"][:150]}')
    print()

SegmentDescription fixed:
  Row count unchanged: 53,585 OK
  Still empty        : 0

Token distribution:
  p25: 36 tokens
  p50: 72 tokens
  p75: 112 tokens
  p90: 150 tokens
  p99: 180 tokens

Samples:
  [20525040] imputed=False
  SD: frozen and vegetables segment includes the green giant and le sueur brands.

  [20525040] imputed=False
  SD: meals segment includes, among others, the ortega, maple grove farms, cream of wheat, las palmas, victoria, mama mary's, spring tree, mccann's, carey's

  [20525040] imputed=False
  SD: specialty segment includes, among others, the crisco, clabber girl, bear creek, polaner, underwood, b and g, grandma's, new york style, don pepino, sc



In [13]:
# ── Phase 4: Revenue Feature Engineering ──────────────────
len_before = len(df_t1)

df_t1['Revenue']                     = df_t1['Revenue'].fillna(0.0).clip(lower=0)
df_t1['total_revenue_company_as_of'] = df_t1['total_revenue_company_as_of'].fillna(0.0).clip(lower=0)
df_t1['revenue_share']               = df_t1['revenue_share'].fillna(0.0).clip(0, 1)

df_t1['log_revenue']       = np.log1p(df_t1['Revenue'])
df_t1['log_total_revenue'] = np.log1p(df_t1['total_revenue_company_as_of'])

assert df_t1['log_revenue'].isnull().sum() == 0
assert df_t1['log_total_revenue'].isnull().sum() == 0

df_t1['is_largest_bin'] = df_t1['is_largest_share_segment'].replace(
    {'TRUE': 1.0, 'FALSE': 0.0, 'True': 1.0, 'False': 0.0,
     True: 1.0, False: 0.0, '1': 1.0, '0': 0.0}
).fillna(0.0).astype(float)

df_t1['year']           = df_t1['AsOfDate'].dt.year
df_t1['report_quarter'] = df_t1['AsOfDate'].dt.quarter

df_t1['n_segments'] = df_t1.groupby('CompanyId')['MstarGlobal'].transform('count')
df_t1['herfindahl_index'] = (
    df_t1['revenue_share'].clip(0, 1) ** 2
).groupby(df_t1['CompanyId']).transform('sum')

assert len(df_t1) == len_before

print('Feature engineering complete:')
print(f'  log_revenue       : nulls={df_t1["log_revenue"].isnull().sum()} ✓')
print(f'  n_segments        : mean={df_t1["n_segments"].mean():.1f}  max={df_t1["n_segments"].max()}')
print(f'  herfindahl_index  : mean={df_t1["herfindahl_index"].mean():.3f}')
print(f'  is_largest_bin    : mean={df_t1["is_largest_bin"].mean():.3f}')
print(f'  Row count         : {len(df_t1):,} ✓')

Feature engineering complete:
  log_revenue       : nulls=0 ✓
  n_segments        : mean=3.6  max=32
  herfindahl_index  : mean=0.610
  is_largest_bin    : mean=0.437
  Row count         : 53,585 ✓


In [14]:
# ── Phase 5: Drop & Deduplicate ───────────────────────────
print(f'Before drops: {len(df_t1):,}')

null_target = df_t1['MstarGlobal'].isnull()
print(f'  Null MstarGlobal          : {null_target.sum():,}')
df_t1 = df_t1[~null_target].copy()

empty_desc = df_t1['SegmentDescription'].str.strip().eq('')
print(f'  Empty SegmentDescription  : {empty_desc.sum():,}')
df_t1 = df_t1[~empty_desc].copy()

before_dedup = len(df_t1)
df_t1 = df_t1.drop_duplicates(
    subset=['CompanyId', 'AsOfDate', 'SegmentName', 'MstarGlobal']
).copy()
print(f'  Exact duplicates dropped  : {before_dedup - len(df_t1):,}')
print(f'\nAfter drops: {len(df_t1):,}')
print(f'Classes    : {df_t1["MstarGlobal"].nunique()} (spec: 145)')

Before drops: 53,585
  Null MstarGlobal          : 0
  Empty SegmentDescription  : 0
  Exact duplicates dropped  : 1,205

After drops: 52,380
Classes    : 145 (spec: 145)


In [15]:
# ── Phase 6: Canonical Split ──────────────────────────────
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
t1_train_idx, t1_test_idx = next(splitter.split(
    df_t1['SegmentDescription'], df_t1['MstarGlobal'], groups=df_t1['CompanyId']
))

train_cos = set(df_t1.iloc[t1_train_idx]['CompanyId'])
test_cos  = set(df_t1.iloc[t1_test_idx]['CompanyId'])
assert len(train_cos & test_cos) == 0, 'Company leakage!'

np.savez_compressed(
    SPLITS_FILE,
    t1_train_idx=t1_train_idx,
    t1_test_idx=t1_test_idx,
)

print('Canonical split saved:')
print(f'  Train : {len(t1_train_idx):,}  Test: {len(t1_test_idx):,}')
print(f'  Overlap: 0 ✓')

Canonical split saved:
  Train : 41,657  Test: 10,723
  Overlap: 0 ✓


In [16]:
# ── Phase 7: Validate & Save ──────────────────────────────
print('=== FINAL VALIDATION ===')
print(f'Classes : {df_t1["MstarGlobal"].nunique()} {"✓" if df_t1["MstarGlobal"].nunique()==145 else "⚠"}')

NUMERIC_COLS = [
    'revenue_share', 'is_largest_bin', 'log_revenue',
    'log_total_revenue', 'n_segments', 'herfindahl_index',
    'report_quarter', 'lp_short_flag', 'sd_short_flag', 'sn_short_flag',
]
print('\nFeature nulls:')
for col in NUMERIC_COLS:
    n = df_t1[col].isnull().sum()
    print(f'  {col:25s} : {n} {"✓" if n==0 else "⚠"}')

print('\nSegmentDescription token distribution:')
for pct in [25, 50, 75, 90, 99]:
    print(f'  p{pct:2d}: ~{df_t1["token_est"].quantile(pct/100):.0f} tokens')

print('\nSamples:')
for _, row in df_t1.sample(3, random_state=42).iterrows():
    print(f'  [{row["MstarGlobal"]}] imputed={row["segment_desc_imputed"]}')
    print(f'  SD: {row["SegmentDescription"][:200]}')
    print()

# Save
KEEP_COLS = [
    'CompanyId', 'AsOfDate', 'MstarGlobal',
    'LongProfile', 'SegmentName', 'SegmentDescription',
    'segment_desc_imputed', 'seg_name_generic', 'weak_signal_row',
    'lp_short_flag', 'sd_short_flag', 'sn_short_flag',
    'Revenue', 'total_revenue_company_as_of',
    'revenue_share', 'is_largest_share_segment',
    'log_revenue', 'log_total_revenue', 'is_largest_bin',
    'n_segments', 'herfindahl_index',
    'year', 'report_quarter', 'seg_desc_len', 'token_est',
]
KEEP_COLS = [c for c in KEEP_COLS if c in df_t1.columns]
df_out    = df_t1[KEEP_COLS].copy()

OUT_PATH  = OUTPUT_DIR / 'task1_gecs_cleaned_v10.csv'
df_out.to_csv(OUT_PATH, index=False)

print(f'✓ Saved: {OUT_PATH}')
print(f'  Shape : {df_out.shape}')
print(f'  Size  : {OUT_PATH.stat().st_size/1e6:.1f} MB')
print()
print('=== v10 CLEANING COMPLETE ===')
print('SegmentDescription = target text column (no ModelInput blob)')
print('All features in separate columns — pulled directly in Dataset class')
print('Next: run 03_flangbert_v10_colab.ipynb')

=== FINAL VALIDATION ===
Classes : 145 ✓

Feature nulls:
  revenue_share             : 0 ✓
  is_largest_bin            : 0 ✓
  log_revenue               : 0 ✓
  log_total_revenue         : 0 ✓
  n_segments                : 0 ✓
  herfindahl_index          : 0 ✓
  report_quarter            : 0 ✓
  lp_short_flag             : 0 ✓
  sd_short_flag             : 0 ✓
  sn_short_flag             : 0 ✓

SegmentDescription token distribution:
  p25: ~36 tokens
  p50: ~72 tokens
  p75: ~112 tokens
  p90: ~150 tokens
  p99: ~180 tokens

Samples:
  [10130020] imputed=False
  SD: performance chemicals business maintains sales and manufacturing capabilities in the united states, canada, europe, south america and australasia. the primary products supplied by pc are copper-based 

  [31040010] imputed=True
  SD: environmental protection (ep) construction engineering services the company is engaged in the development, manufacture and sale of ep products and equipment, and the provision of ep constructio

In [3]:
import pandas as pd
t2 = pd.read_csv('../data/raw/task2_subindustry_classification_final.csv', nrows=5)
print(t2.columns.tolist())
print(t2.shape)
print(t2.head(2).to_string())

['CompanyId', 'AsOfDate', 'SegmentName', 'SegmentDescription', 'SubIndustry']
(5, 5)
    CompanyId    AsOfDate              SegmentName                                                                                                                                      SegmentDescription  SubIndustry
0  3IZBDS5MVU  2020-12-31               Substrates  Substrates engages in the design, development, manufacture, and distribution of high-performance compound and single element semiconductor substrates.   3113001001
1  3IZBDS5MVU  2020-12-31  Raw Materials and Other                                               Raw Materials and Other pertains to the sale of raw materials integral to producing the substrate wafers.   3113001001
